# Trimorious M1.2.1 — Scientific Dataset Reconstruction
**Option A · separate candidates · CPU only · no publishing or training**

This version fixes the nested `messages` schema discovered in the M1.2 audit. It reconstructs a small set of independently computed mathematics candidates, rather than retaining unrelated research context or inflating the number of unique tasks. D03 is retrieval-only; D04 replay evidence is not SFT data. All records have a disposition and source fingerprint. Unknown schemas or unsupported equations are never silently accepted.

Run cells in order in Google Colab. Default input is the public Hugging Face datasets, pinned to resolved commit SHAs. Alternatively choose the previous audit ZIP. No token is needed for public data. No network writes are implemented. Use a fresh runtime for each run.

In [ ]:
%pip -q install datasets huggingface_hub pyarrow sympy pandas

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
from functools import lru_cache
import ast, csv, hashlib, io, json, re, tokenize, zipfile, platform
import sympy as sp

SOURCE_MODE = "huggingface"  # or "audit_zip"
AUDIT_ZIP = "/content/trimorious_m1_2_audit_bundle_m12_20261126T142337Z.zip"
REPOS = {
 "D01":"roime81/trimorious-scientific-instructions",
 "D02":"roime81/trimorious-symbolic-trajectories",
 "D03":"roime81/trimorious-knowledge-base",
 "D04":"roime81/trimorious-agentic-reasoning-knowledge"
}
HISTORICAL_COUNTS = {"D01":3706,"D02":7784,"D03":7784,"D04":225}
# Optional explicit SHA overrides to reproduce a prior run.
REVISION_OVERRIDES = {}
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT = Path("/content/trimorious_m1_2_1") / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
ALLOW_TRAINING = False
ALLOW_PUBLISH = False
print(OUT)

## 1 — Read-only ingestion and provenance
Hugging Face mode loads all splits of the single configuration per dataset. Multiple configurations require an explicit loader change rather than silently selecting one. Audit mode recovers the preserved raw JSON, including NumPy-style string representations of nested messages. The audit archive does not establish the original upstream commit SHA. The old 596-row discrepancy cannot be explained from the current audit alone.

In [ ]:
def sha(value):
    return hashlib.sha256(value.encode('utf-8')).hexdigest()
def canon(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(',',':'), allow_nan=False)
def file_hash(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()

def read_inputs():
    rows={k:[] for k in REPOS}; revisions={}
    if SOURCE_MODE == 'huggingface':
        from huggingface_hub import HfApi
        from datasets import load_dataset, get_dataset_config_names
        api=HfApi()
        for k,repo in REPOS.items():
            commit=api.dataset_info(repo,revision=REVISION_OVERRIDES.get(k)).sha
            if not commit: raise RuntimeError(f'{k}: missing commit SHA')
            configs=get_dataset_config_names(repo,revision=commit)
            if len(configs)!=1: raise RuntimeError(f'{k}: specify configuration explicitly: {configs}')
            ds=load_dataset(repo,name=configs[0],revision=commit)
            for split,data in ds.items():
                for record in data:
                    rows[k].append(dict(record,_source_split=split))
            revisions[k]={'repo':repo,'sha':commit,'config':configs[0]}
    elif SOURCE_MODE == 'audit_zip':
        import pandas as pd
        csv.field_size_limit(10_000_000)
        with zipfile.ZipFile(AUDIT_ZIP) as z:
            with z.open('reports/human_review_queue.csv') as f:
                for record in csv.DictReader(io.TextIOWrapper(f,encoding='utf8')):
                    k=record['source_dataset']
                    if k not in ('D01','D02','D04'): raise ValueError(k)
                    rows[k].append(json.loads(record['raw_record_json']))
            df=pd.read_parquet(io.BytesIO(z.read('candidate_datasets/D03_retrieval_candidate_unapproved.parquet')))
            rows['D03']=[json.loads(s) for s in df['raw_record_json']]
        revisions={k:{'repo':v,'sha':None,'audit_zip_sha256':file_hash(AUDIT_ZIP)} for k,v in REPOS.items()}
    else: raise ValueError(SOURCE_MODE)
    for k,rs in rows.items():
        if not rs: raise RuntimeError(f'{k} is empty; stop')
        print(k,len(rs),'historical',HISTORICAL_COUNTS[k])
    return rows,revisions

In [ ]:
rows, revisions = read_inputs()

## 2 — Schema-aware parser and safety tests
For HF, messages are structured lists. For old audit strings, insert missing commas only at token boundaries, then use Python literal parsing (never `eval`). The only math inputs that will be computed are predefined, bounded expressions in a registry. No source-provided string is executed or passed to `sympify`, `parse_expr`, or a shell. Unknown cases remain quarantined.

In [ ]:
def parse_messages(value):
    if isinstance(value,str):
        if len(value)>100_000: raise ValueError('message representation too large')
        try: value=json.loads(value)
        except json.JSONDecodeError:
            # NumPy repr uses adjacent dicts separated by whitespace, without commas.
            ts=list(tokenize.generate_tokens(io.StringIO(value).readline))
            significant=[]; last=None
            for t in ts:
                if t.type in (tokenize.NL,tokenize.NEWLINE,tokenize.ENDMARKER,tokenize.INDENT,tokenize.DEDENT): continue
                if last=='}' and t.string=='{': significant.append((tokenize.OP,','))
                significant.append((t.type,t.string)); last=t.string
            value=ast.literal_eval(tokenize.untokenize(significant))
    if hasattr(value,'tolist'): value=value.tolist()
    if not isinstance(value,list) or not value: raise ValueError('messages must be a list')
    if any(not isinstance(v,dict) or not isinstance(v.get('role'),str) or not isinstance(v.get('content'),str) for v in value):
        raise ValueError('invalid message item')
    # Only the observed three-turn layout is supported. No multi-turn data silently dropped.
    if [v['role'] for v in value] != ['system','user','assistant']:
        raise ValueError('unsupported conversation structure')
    if any(not v['content'].strip() for v in value): raise ValueError('empty content')
    return [{'role':v['role'],'content':v['content']} for v in value]

assert parse_messages("[{'role':'system','content':'s'}\n {'role':'user','content':'q'}\n {'role':'assistant','content':'a'}]")[1]['content']=='q'
try:
    parse_messages("__import__('os').system('echo unsafe')")
except (ValueError,SyntaxError,TypeError): pass
else: raise AssertionError('unsafe literal accepted')

## 3 — Verified expression registry
This registry covers the exact 3 D01, 4 D02 and 12 D04 expression templates identified in the uploaded audit. It is intentionally narrow. It computes derivatives using SymPy objects built from trusted code and checks the final source LaTeX against the computed canonical LaTeX (ignoring whitespace only). Equivalent alternate LaTeX not recognized by this check is quarantined, not judged incorrect. D04 additionally checks its recorded tool call, observation and ground truth using an exact allowlist for this specific derivative family.

In [ ]:
x=sp.Symbol('x')
REG={}
def add(k, raw, expr, family, explanation, domain):
    REG[(k,raw)]={'expr':expr,'family':family,'explanation':explanation,'domain':domain}
add('D01',r'\frac{\sin(x)}{x}',sp.sin(x)/x,'quotient','Use the quotient rule with numerator sin(x), derivative cos(x), and denominator x, derivative 1.','real x ≠ 0')
add('D01',r'x^2 * e^{-3x}',x**2*sp.exp(-3*x),'product_exponential','Use the product rule: the derivatives of x² and exp(-3x) are 2x and -3 exp(-3x).','all real x')
add('D01',r'e^{3x} \cdot \sin(x)',sp.exp(3*x)*sp.sin(x),'product_exponential_trig','Use the product and chain rules: the derivatives of exp(3x) and sin(x) are 3 exp(3x) and cos(x).','all real x')
add('D02','exp(-x)*sin(2*x)',sp.exp(-x)*sp.sin(2*x),'product_exponential_trig','Apply the product rule with derivatives -exp(-x) and 2 cos(2x).','all real x')
add('D02','x**3*cos(x)',x**3*sp.cos(x),'product_polynomial_trig','Apply the product rule with derivatives 3x² and -sin(x).','all real x')
add('D02','log(x**2 + 1)/x',sp.log(x**2+1)/x,'quotient','Use the quotient and chain rules; the derivative of ln(x²+1) is 2x/(x²+1).','real x ≠ 0')
add('D02','3*x**2*exp(-5*x)',3*x**2*sp.exp(-5*x),'product_exponential','Apply the product and chain rules; the derivatives of 3x² and exp(-5x) are 6x and -5 exp(-5x).','all real x')
for p in (2,3,4):
    for a in (2,3,4,5):
        expr=x**p*sp.sin(a*x)
        add('D04',sp.latex(expr),expr,'product_polynomial_trig',f'Apply the product and chain rules with derivatives {p}x^{p-1} and {a} cos({a}x).','all real x')
        REG[('D04',sp.latex(expr))].update(p=p,a=a)
for item in REG.values():
    item['derivative']=sp.diff(item['expr'],x)
assert len(REG)==19

def compact(s): return re.sub(r'\s+','',s)
def verify_math(k,user,answer,raw):
    match=re.search(r'f\(x\)\s*=\s*(.*?)\s+with respect to x',user,re.S)
    if not match: return None,'no_supported_derivative_prompt',{}
    expr_text=match.group(1).strip(); item=REG.get((k,expr_text))
    if item is None: return None,'unrecognized_expression',{'source_expression':expr_text}
    rhs=re.findall(r'\$\$\\frac\{df\}\{dx\}\s*=\s*(.*?)\$\$',answer,re.S)
    expected=sp.latex(item['derivative'])
    check={'source_expression':expr_text,'expected_latex':expected,'source_final_latex':rhs[-1] if rhs else '',
           'family':item['family'],'domain':item['domain']}
    if not rhs or compact(rhs[-1])!=compact(expected): return None,'source_final_latex_not_matched',check
    if k=='D04':
        calls=re.findall(r'<call:calc_sympy>(.*?)</call:calc_sympy>',answer,re.S)
        obs=re.findall(r'<observation>(.*?)</observation>',answer,re.S)
        p,a=item['p'],item['a']
        call=f'diff(sin({a}*x) * x**{p}, x)'
        expected_text=str(item['derivative'])
        check.update(recorded_calls=calls,recorded_observations=obs,ground_truth=raw.get('ground_truth'),replayed_result=expected_text)
        # Validate ordering and exact call shape; recomputation is from the trusted registry, not the text call.
        if len(calls)!=1 or len(obs)!=1 or answer.index('<call:calc_sympy>')>answer.index('<observation>') or compact(calls[0])!=compact(call):
            return None,'unsupported_or_malformed_tool_call',check
        if compact(obs[0])!=compact(expected_text) or compact(str(raw.get('ground_truth','')))!=compact(expected_text):
            return None,'tool_observation_or_ground_truth_mismatch',check
    return item,'verified_derivative_reference',check

## 4 — Reconstruct candidates and retain a complete disposition ledger
D01/D02: generate new standalone math prompts and concise derivations from the trusted registry. Remove original paper snippets and unsupported physical claims entirely. One candidate per distinct expression across D01/D02. Non-math D01 records remain quarantined for source-grounded rebuilding. D04: replay-check every source record, retain one structured replay candidate per unique expression; no simulated thought traces or original paper claims become training content. D03: retain all document metadata and text, locally only, with license unresolved unless separately reviewed.

In [ ]:
def reconstruct(rows):
    ledger=[]; candidate_by_id={}; retrieval=[]; reviews=[]; replay_by_id={}; seen_raw=set()
    for k in REPOS:
        for i,raw in enumerate(rows[k]):
            raw_hash=sha(canon(raw)); rid=f'{k}:{i:07d}'
            rec={'record_id':rid,'dataset':k,'source_split':raw.get('_source_split','unknown'),'raw_sha256':raw_hash}
            duplicate_key=(k,raw_hash)
            rec['exact_raw_duplicate']=duplicate_key in seen_raw; seen_raw.add(duplicate_key)
            if k=='D03':
                required=('doc_id','title','source','section','text')
                if any(key not in raw or not isinstance(raw[key],str) or not raw[key].strip() for key in ('title','source','text')) or any(key not in raw or raw[key] is None for key in required):
                    rec.update(disposition='quarantine_invalid_retrieval_schema')
                    reviews.append({'record_id':rid,'reason':rec['disposition'],'raw':raw})
                else:
                    retrieval.append({**{key:raw[key] for key in required},'source_record_id':rid,'raw_sha256':raw_hash,'text_sha256':sha(raw['text']),'license_status':'unverified','release_approved':False})
                    rec.update(disposition='retrieval_only_license_review')
                ledger.append(rec); continue
            try: messages=parse_messages(raw.get('messages'))
            except (ValueError,TypeError,SyntaxError,tokenize.TokenError) as e:
                rec.update(disposition='quarantine_schema_error',error=str(e)); ledger.append(rec)
                reviews.append({'record_id':rid,'reason':rec['disposition'],'raw':raw}); continue
            user,answer=messages[1]['content'],messages[2]['content']
            rec['conversation_sha256']=sha(canon(messages))
            rec['question_sha256']=sha(user)
            item,status,check=verify_math(k,user,answer,raw)
            rec['verification']=check
            if item is None:
                rec['disposition']='quarantine_'+status
                reviews.append({'record_id':rid,'reason':rec['disposition'],'raw':raw})
            else:
                expr=item['expr']; result=item['derivative']; cid='math_'+sha(sp.srepr(expr))[:24]
                rec['candidate_id']=cid; rec['template_family']=item['family']
                # Deduplication spans D01 and D02; replay has its own non-training collection.
                collection=replay_by_id if k=='D04' else candidate_by_id
                if cid in collection:
                    rec['disposition']='collapsed_repeated_template'
                    collection[cid]['source_record_ids'].append(rid)
                else:
                    latex=sp.latex(expr); derivative=sp.latex(result)
                    entry={'candidate_id':cid,'source_dataset':k,'source_record_ids':[rid],
                           'family':item['family'],'expression':str(expr),'derivative':str(result),
                           'domain':item['domain'],'verification':'symbolic_derivative_recomputed',
                           'release_approved':False,'training_approved':False}
                    if k=='D04':
                        entry.update(tool='calc_sympy',arguments={'expression':str(expr),'variable':'x'},observation=str(result),
                                     trace_status='recorded_call_and_observation_matched_by_recomputation',
                                     scope='replay_evidence_only')
                        rec['disposition']='replay_candidate_unapproved'
                    else:
                        entry['messages']=[{'role':'system','content':'You are a scientific research assistant. State assumptions and distinguish mathematical results from scientific claims.'},
                        {'role':'user','content':f'Differentiate f(x) = {latex} with respect to x. State the real domain and explain the calculus rule.'},
                        {'role':'assistant','content':f"Domain: {item['domain']}. {item['explanation']}\n\n\\[f'(x) = {derivative}\\]"}]
                        rec['disposition']='rebuilt_math_candidate_unapproved'
                    collection[cid]=entry
            ledger.append(rec)
    return ledger,list(candidate_by_id.values()),retrieval,list(replay_by_id.values()),reviews

In [ ]:
ledger, math_candidates, retrieval, replay_candidates, quarantine = reconstruct(rows)

## 5 — Quality gates and leakage controls
Reconcile every row and each candidate link. No train/validation split is produced: 7 unique math candidates across only a few families cannot establish meaningful generalization. D04 replay evidence is separate and remains unapproved. The D07 benchmark is not generated from these templates. In M1.3 use independent expert-authored questions, source and template-family disjointness, contamination checks, and frozen benchmark manifests before approving a split.

In [ ]:
def validate(rows,ledger,math_candidates,retrieval,replay_candidates,quarantine):
    assert len(ledger)==sum(map(len,rows.values()))
    ids=[r['record_id'] for r in ledger]; assert len(ids)==len(set(ids))
    counts=Counter(r['dataset'] for r in ledger)
    assert counts==Counter({k:len(v) for k,v in rows.items()})
    schema_fail=[r for r in ledger if r['disposition'] in ('quarantine_schema_error','quarantine_invalid_retrieval_schema')]
    candidate_ids=[r['candidate_id'] for r in math_candidates+replay_candidates]
    assert len(candidate_ids)==len(set(candidate_ids)), 'Cross-collection expression overlap: review required'
    linked={rid for item in math_candidates+replay_candidates for rid in item['source_record_ids']}
    assert linked=={r['record_id'] for r in ledger if 'candidate_id' in r}
    assert {r['record_id'] for r in ledger if r['disposition'].startswith('quarantine_')}=={r['record_id'] for r in quarantine}
    assert len(retrieval)==sum(r['disposition']=='retrieval_only_license_review' for r in ledger)
    assert all(not r['release_approved'] for r in math_candidates+retrieval+replay_candidates)
    assert all(not r['training_approved'] for r in math_candidates+replay_candidates)
    return {'schema_failures':len(schema_fail),'counts_reconciled':True,'training_ready':False,'publish_ready':False,
            'candidate_status':'blocked_schema_errors' if schema_fail else 'review_only'}
checks=validate(rows,ledger,math_candidates,retrieval,replay_candidates,quarantine)
summary={k:dict(Counter(r['disposition'] for r in ledger if r['dataset']==k)) for k in REPOS}
print(json.dumps({'dispositions':summary,'math_candidates':len(math_candidates),'retrieval_chunks':len(retrieval),'replay_candidates':len(replay_candidates),'gates':checks},indent=2))

## 6 — Export isolated candidate collections and audit artifacts
The audit ZIP includes quarantined raw records and unpublished document chunks: treat it as private review material, not a redistribution package. No Hugging Face repository is created or updated. Candidate filenames distinguish D01, D02, D03 and D04. Only the review-only outputs are generated; D07 is a blank schema and is not training material.

In [ ]:
def write_jsonl(path,records):
    with open(path,'w',encoding='utf8') as f:
        for r in records: f.write(canon(r)+'\n')
def write_json(path,value): path.write_text(json.dumps(value,indent=2,ensure_ascii=False),encoding='utf8')
for k in ('D01','D02'):
    write_jsonl(OUT/f'{k}_math_reconstructed_review_only.jsonl',[r for r in math_candidates if r['source_dataset']==k])
write_jsonl(OUT/'D03_retrieval_unapproved.jsonl',retrieval)
write_jsonl(OUT/'D04_replay_evidence_NOT_SFT.jsonl',replay_candidates)
write_jsonl(OUT/'quarantine_private_review.jsonl',quarantine)
write_jsonl(OUT/'record_disposition_ledger.jsonl',ledger)
write_json(OUT/'D07_template_DO_NOT_TRAIN.json',{'records':[],'required_fields':['benchmark_id','question','gold_answer_or_rubric','source_id','template_family','verifier','expert_review','contamination_check'],'training_use':False})
write_json(OUT/'summary.json',{'run_id':RUN_ID,'revisions':revisions,'source_counts':{k:len(v) for k,v in rows.items()},'historical_counts':HISTORICAL_COUNTS,'dispositions':summary,'candidate_counts':{'D01':sum(r['source_dataset']=='D01' for r in math_candidates),'D02':sum(r['source_dataset']=='D02' for r in math_candidates),'D03':len(retrieval),'D04':len(replay_candidates)},'gates':checks,'python':platform.python_version(),'sympy':sp.__version__,'historical_596_cause':'unresolved'})
write_json(OUT/'manifest.json',{p.name:file_hash(p) for p in sorted(OUT.iterdir()) if p.is_file()})
zip_path=OUT.parent/f'trimorious_m1_2_1_reconstruction_{RUN_ID}.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file(): z.write(p,p.name)
print(zip_path)

In [ ]:
# Colab download; this does not publish anything.
from google.colab import files
files.download(str(zip_path))

## 7 — Exit criteria and next milestone
- Review summary, ledger, quarantine and all reconstructed examples.
- If schema failures appear, fix ingestion before candidate approval.
- Confirm source/license provenance for D03; never redistribute merely because text is accessible.
- Rebuild scientific methodology and source-grounded reasoning from authorized evidence, not generic templates.
- Develop D07 independently before large-scale data augmentation.
- Keep original Hugging Face datasets and weights immutable. No SFT, DPO or GRPO until benchmark and dataset quality gates are met.